# LABORATORIO DIRIGIDO N.° 03
**Semana 3 · SQL Avanzado — Joins y Agrupaciones**  
Clientes, pedidos y productos de TecnoAndina Retail con JOIN, agrupaciones y vistas en SQLite  
Curso: Fundamentos de Gestión de Datos · Periodo: 2026-II · Docente: Pilar Rocío Sayán Mejía  
Datos: `LD3_TecnoAndina_Retail.xlsx` (GitHub: Rociosayan/Repo_FDG_2026-2, carpeta semana03/datos/)

### Nombre
Erick Pariona Yauricasa

## Caso de estudio: TecnoAndina Retail

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web. Su información está repartida en cuatro tablas y el área comercial necesita saber qué clientes compran más, cuáles todavía no compran y qué productos y categorías sustentan las ventas.

Naturaleza de los datos. Conjunto de práctica inventado (no son ventas ni personas reales), con pedidos del 1 de agosto al 30 de septiembre de 2026, publicado en GitHub en LD3_TecnoAndina_Retail.xlsx. Sirve para practicar consultas, no para afirmar tendencias ni rentabilidad.

- clientes: id_cliente, nombre, ciudad. Una fila es un cliente registrado.

- productos: id_producto, nombre_producto, categoria, precio_lista. Una fila es un producto del catálogo.

- pedidos: id_pedido, id_cliente, fecha_pedido, canal, total. Una fila es un pedido de un cliente.

- detalles_pedido: id_detalle, id_pedido, id_producto, cantidad, precio_unitario. Una fila es un producto incluido en un pedido.

Las tablas se relacionan por id_cliente (clientes–pedidos), id_pedido (pedidos–detalles_pedido) e id_producto (productos–detalles_pedido).

### Preparación del entorno

Abra el notebook en Google Colab y ejecute la celda 1: importa sqlite3 y pandas, crea la base en memoria, carga las cuatro hojas del Excel desde GitHub y cuenta los registros. Cada resultado se muestra con pd.read_sql_query; pandas muestra los NULL como NaN o None. Si reinicia el entorno, vuelva a ejecutar la celda 1.

**Celda 1 de 12**

In [24]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail (agosto y septiembre de 2026)
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"
conexion = sqlite3.connect(":memory:")

# Cada hoja del Excel pasa a una tabla de SQLite
pd.read_excel(url, sheet_name="clientes").to_sql("clientes", conexion, index=False)
pd.read_excel(url, sheet_name="productos").to_sql("productos", conexion, index=False)
pd.read_excel(url, sheet_name="pedidos").to_sql("pedidos", conexion, index=False)
pd.read_excel(url, sheet_name="detalles_pedido").to_sql("detalles_pedido", conexion, index=False)

consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes,
  (SELECT COUNT(*) FROM productos) AS productos,
  (SELECT COUNT(*) FROM pedidos) AS pedidos,
  (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
pd.read_sql_query(consulta, conexion)

,clientes,productos,pedidos,detalles_pedido
0,24,14,42,68


## A. Relacionar tablas

### Ejercicio 1 — INNER JOIN y LEFT JOIN: clientes con y sin pedidos

El área comercial quiere saber cuántos clientes compraron y quiénes todavía no registran pedidos. INNER JOIN conserva solo las filas con pareja; LEFT JOIN conserva todas las filas de la tabla izquierda y completa con NULL.

Se solicita. Ejecute las celdas 2 y 3.

**Celda 2 de 12**

In [3]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,pedidos_unidos,clientes_con_pedidos,clientes_registrados
0,42,18,24


**Celda 3 de 12**

In [8]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad
0,6,Andrés Villanueva,Lima
1,11,Raúl Benavides,Arequipa
2,15,Héctor Burga,Trujillo
3,20,Marta Infante,Piura
4,22,Gabriela Ponce,Arequipa
5,24,Natalia Ato,Piura


**Pregunta 1.** ¿Cuántos pedidos y cuántos clientes distintos devuelve la celda 2? ¿Por qué INNER JOIN no muestra a todos los clientes registrados?

**Respuesta:** La celda 2 devuelve 42 pedidos y 18 clientes distintos con pedidos, de un total de 24 clientes registrados. INNER JOIN no muestra a todos los clientes porque únicamente relaciona los registros que tienen coincidencia en el campo id_cliente de ambas tablas. Por ello, los 6 clientes que no realizaron pedidos quedan excluidos del resultado.

**Pregunta 2.** En la celda 3, ¿qué tabla es la izquierda y por qué se filtra con p.id_pedido IS NULL y no con = NULL? ¿Cuántos clientes no tienen pedidos?

**Respuesta:** La tabla izquierda es clientes (c), porque aparece antes del LEFT JOIN. Se utiliza p.id_pedido IS NULL para identificar a los clientes que no tienen pedidos asociados, ya que LEFT JOIN conserva todos los registros de la tabla clientes y devuelve NULL en los campos de pedidos cuando no encuentra coincidencias. No se utiliza = NULL porque en SQL los valores nulos se comprueban mediante IS NULL. Según el resultado de la celda 3, existen 6 clientes que no tienen pedidos registrados.

### Ejercicio 2 — RIGHT JOIN y FULL OUTER JOIN

El área de Compras pregunta:¿qué productos no se vendieron y la jefatura quiere comparar a los clientes entre agosto y septiembre. RIGHT JOIN conserva la tabla derecha; FULL OUTER JOIN conserva las filas de ambas tablas. Ambos requieren SQLite 3.39.0 o posterior.

Se solicita. Ejecute la celda 4 (versión y RIGHT JOIN) y la celda 5, que crea un resumen por cliente para cada mes y los une con FULL OUTER JOIN.

**Celda 4 de 12**

In [9]:
# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
display(pd.read_sql_query(consulta, conexion))

,version_sqlite
0,3.45.1


,id_producto,nombre_producto,categoria
0,5,Impresora multifuncional,Cómputo
1,14,Reloj inteligente,Accesorios


**Celda 5 de 12**

In [10]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.monto AS monto_agosto, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,monto_agosto,monto_septiembre
0,1,1435.0,1967.0
1,2,1522.0,2615.0
2,3,656.0,840.0
3,4,2452.0,NaN
4,5,NaN,640.0
5,7,165.0,3460.0
6,8,271.0,236.0
7,9,220.0,NaN
8,10,6400.0,723.0
9,12,NaN,1420.0


**Pregunta 3.** ¿Qué versión de SQLite informó la celda 4? En el RIGHT JOIN, ¿qué tabla es la derecha y qué productos no se vendieron?

**Respuesta:** La celda 4 informa que la versión de SQLite es 3.45.1. En el RIGHT JOIN, la tabla derecha es productos (pr), por lo que se conservan todos los productos, incluso aquellos que no tienen ventas registradas. Los productos que no se vendieron son Impresora multifuncional (id_producto 5, categoría Cómputo) y Reloj inteligente (id_producto 14, categoría Accesorios).

**Pregunta 4.** En la celda 5, ¿cuántos clientes compraron solo en agosto, solo en septiembre y en ambos meses? ¿Cómo los reconoce por los NULL?

**Respuesta:** La celda 5 devuelve 18 clientes distintos: 3 compraron solo en agosto, 3 compraron solo en septiembre y 12 compraron en ambos meses. Se identifican mediante los valores NULL, que pandas muestra como NaN. Si monto_septiembre es NaN, significa que el cliente compró únicamente en agosto; si monto_agosto es NaN, compró únicamente en septiembre. Cuando ambas columnas tienen valores numéricos, significa que realizó compras en ambos meses.

## B. Agrupar y filtrar grupos

### Ejercicio 3 — WHERE, GROUP BY y HAVING

La jefatura pide los clientes con al menos dos pedidos en septiembre. WHERE filtra filas antes de agrupar; GROUP BY forma un grupo por cliente; HAVING filtra los grupos ya formados.

Se solicita. Ejecute la celda 6.

**Celda 6 de 12**

In [11]:
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre,
       SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,pedidos_septiembre,monto_septiembre
0,2,Carlos Mendoza,3,2615
1,1,Lucía Paredes,2,1967
2,12,Sofía Linares,2,1420
3,17,Luis Távara,2,885
4,10,Patricia Zevallos,2,723


**Pregunta 5.** ¿Qué filtra WHERE y qué filtra HAVING? Un cliente con tres pedidos en total, pero uno solo en septiembre, ¿aparece? Justifique.

**Respuesta:** WHERE filtra los pedidos realizados desde el 1 de septiembre de 2026, antes de agrupar los registros. HAVING filtra los grupos de clientes después de aplicar GROUP BY, mostrando únicamente aquellos que realizaron dos o más pedidos en septiembre. Un cliente que tiene tres pedidos en total, pero solo uno en septiembre, no aparece en el resultado, porque WHERE excluye los pedidos anteriores a septiembre y HAVING exige que el cliente tenga al menos dos pedidos durante ese mes.

## C. Consultar y reutilizar resultados

### Ejercicio 4 — Subconsulta: pedidos sobre el promedio por pedido

Logística quiere revisar los pedidos que superan el monto de un pedido promedio. La subconsulta calcula primero el promedio; la consulta exterior compara cada pedido con ese valor.

Se solicita. Ejecute la celda 7.

**Celda 7 de 12**

In [12]:
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

,id_pedido,fecha_pedido,id_cliente,nombre,total
0,3,2026-08-02,10,Patricia Zevallos,6400
1,13,2026-08-17,13,Diego Alva,5655
2,26,2026-09-08,7,Carlos Mendoza,3460
3,31,2026-09-15,18,Ana Girón,3045
4,35,2026-09-24,2,Carlos Mendoza,2544
5,20,2026-08-21,4,Jorge Salazar,2452
6,9,2026-08-12,16,Camila Neyra,2256
7,15,2026-08-18,17,Luis Távara,1680
8,11,2026-08-16,2,Carlos Mendoza,1522
9,28,2026-09-10,1,Lucía Paredes,1485


**Pregunta 7.** ¿Qué devuelve la subconsulta y cuántos pedidos la superan? ¿Qué clientes aparecen más de una vez?

**Respuesta:** La subconsulta devuelve el promedio del monto total de todos los pedidos registrados en la tabla pedidos, utilizando la función AVG(total). La consulta principal muestra únicamente los pedidos cuyo monto supera ese promedio. Según los resultados de la celda 7, existen 13 pedidos que superan el promedio. Los clientes que aparecen más de una vez son Carlos Mendoza, con 3 pedidos, y Lucía Paredes, con 2 pedidos.

### Ejercicio 5 — Vista: guardar el resumen por cliente

Varias áreas piden el mismo resumen por cliente. Una vista guarda la consulta con un nombre y se vuelve a ejecutar cada vez que se consulta.

Se solicita. Ejecute la celda 8, que crea vista_resumen_cliente y la consulta.

**Celda 8 de 12**

In [25]:
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
""")
consulta = """SELECT *
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,7,Carlos Mendoza,Arequipa,2,3625,1812.50
1,2,Carlos Mendoza,Lima,4,4137,1034.25
2,17,Luis Távara,Piura,4,3224,806.00
3,18,Ana Girón,Piura,1,3045,3045.00
4,19,Pedro Seminario,Piura,2,1502,751.00
5,20,Marta Infante,Piura,0,0,NaN
6,24,Natalia Ato,Piura,0,0,NaN


**Pregunta 8.** ¿Qué guarda la vista: los datos o la consulta? ¿Qué columna distingue a los dos clientes llamados Carlos Mendoza?

**Respuesta:** La vista guarda la consulta SQL con el nombre vista_resumen_cliente, no los datos físicamente. Cada vez que se consulta, obtiene la información actualizada de las tablas clientes y pedidos. La columna que distingue a los dos clientes llamados Carlos Mendoza es id_cliente, ya que uno tiene el identificador 7 y pertenece a Arequipa, mientras que el otro tiene el identificador 2 y pertenece a Lima. Esto permite identificar correctamente a cada cliente, aunque tengan el mismo nombre.

## Actividad 3 — Informe analítico comparado

La gerencia evalúa ofrecer beneficios a los clientes que más compraron. Ejecute la celda 9, complete la tabla y responda citando cifras. Liderar por monto en dos meses no convierte a un cliente en el más rentable.

**Celda 9 de 12**

In [22]:
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,10,Patricia Zevallos,Arequipa,3,7123,2374.33
1,13,Diego Alva,Trujillo,2,5879,2939.50
2,2,Carlos Mendoza,Lima,4,4137,1034.25
3,7,Carlos Mendoza,Arequipa,2,3625,1812.50
4,1,Lucía Paredes,Lima,4,3402,850.50


### Tabla: Clientes con mayor monto acumulado

| Posición | id_cliente | Nombre | Ciudad | Pedidos | Monto acumulado (S/) | Promedio por pedido (S/) |
|---|---|---|---|---|---|---|
| 1 | 10 | Patricia Zevallos | Arequipa | 3 | 7123.00 | 2374.33 |
| 2 | 13 | Diego Alva | Trujillo | 2 | 5879.00 | 2939.50 |
| 3 | 2 | Carlos Mendoza | Lima | 4 | 4137.00 | 1034.25 |
| 4 | 7 | Carlos Mendoza | Arequipa | 2 | 3625.00 | 1812.50 |
| 5 | 1 | Lucía Paredes | Lima | 4 | 3402.00 | 850.50 |

**A.** ¿Qué clientes lideran por monto acumulado y quién tiene el mayor promedio por pedido?

**Respuesta:** Patricia Zevallos lidera por monto acumulado con S/ 7123.00 en 3 pedidos, seguida de Diego Alva con S/ 5879.00 en 2 pedidos y Carlos Mendoza de Lima con S/ 4137.00 en 4 pedidos. Sin embargo, Diego Alva tiene el mayor promedio por pedido, con S/ 2939.50, superando a Patricia Zevallos, cuyo promedio es S/ 2374.33.

**B.** ¿Qué información adicional pediría antes de recomendar beneficios? Redacte una recomendación prudente en tres líneas.

**Respuesta:** Solicitaría información sobre el margen de ganancia, la frecuencia de compra, las devoluciones y los costos asociados a cada cliente.
Recomendaría evaluar beneficios para los clientes con mayores compras, considerando también la rentabilidad que generan para la empresa.
Antes de implementar descuentos o promociones, analizaría su impacto económico para evitar afectar las ganancias del negocio.

## Reto de aplicación y retroalimentación

Indicaciones. Cada reto presenta la consulta ya escrita: ejecútela, identifique qué hace cada cláusula e interprete el resultado en no más de cuatro líneas. Los retos son la base de la actividad calificada de la semana.

### Reto 1 — Clientes sin pedidos por ciudad

Marketing quiere saber en qué ciudades se concentran los clientes que todavía no tienen pedidos. Un cliente sin pedidos no es necesariamente un cliente perdido.

**Celda 10 de 12**

In [14]:
consulta = """SELECT c.ciudad, COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

,ciudad,clientes_sin_pedidos
0,Arequipa,2
1,Piura,2
2,Lima,1
3,Trujillo,1


**Pregunta de cierre.** ¿Cuántos clientes sin pedidos hay y en qué ciudades se concentran? ¿Qué cláusula los conserva y cuál los aísla?

**Respuesta:** Existen 6 clientes sin pedidos registrados, distribuidos en Arequipa (2), Piura (2), Lima (1) y Trujillo (1). La mayor concentración se encuentra en Arequipa y Piura, con 2 clientes cada una. La cláusula LEFT JOIN conserva todos los clientes, incluso aquellos que no tienen pedidos asociados, mientras que WHERE p.id_pedido IS NULL permite aislar únicamente a los clientes sin pedidos.

### Reto 2 — Ingresos por categoría con un umbral

Finanzas quiere saber qué categorías generaron al menos S/ 10 000 entre agosto y septiembre. El ingreso se calcula desde las líneas de detalle.

**Celda 11 de 12**

In [21]:
consulta = """SELECT pr.categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
pd.read_sql_query(consulta, conexion)

,categoria,unidades,ingreso
0,Celulares,12,18805
1,Cómputo,37,17662


**Pregunta de cierre.** ¿Qué categorías superan el umbral y cuál queda fuera? ¿Por qué la condición va en HAVING y no en WHERE?

**Respuesta:** Las categorías que superan el umbral de S/ 10 000 son Celulares, con ingresos de S/ 18 805 y 12 unidades vendidas, y Cómputo, con S/ 17 662 y 37 unidades vendidas. La categoría Accesorios queda fuera porque no alcanza el ingreso mínimo establecido. La condición se utiliza en HAVING porque permite filtrar los resultados después de agruparlos por categoría y calcular la suma de ingresos. No se utiliza WHERE porque esta cláusula filtra los registros antes de realizar las agrupaciones.

### Reto 3 — Clientes sobre el promedio por cliente

Comercial quiere identificar a los clientes cuyo monto acumulado supera el promedio de los clientes que compraron. La consulta usa la vista de la celda 8 y una subconsulta.

**Celda 12 de 12**

In [26]:
consulta = """SELECT id_cliente, nombre, ciudad, monto_acumulado
FROM vista_resumen_cliente
WHERE monto_acumulado > (SELECT AVG(monto_acumulado)
                         FROM vista_resumen_cliente
                         WHERE num_pedidos > 0)
ORDER BY monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,monto_acumulado
0,10,Patricia Zevallos,Arequipa,7123
1,13,Diego Alva,Trujillo,5879
2,2,Carlos Mendoza,Lima,4137
3,7,Carlos Mendoza,Arequipa,3625
4,1,Lucía Paredes,Lima,3402
5,17,Luis Távara,Piura,3224
6,18,Ana Girón,Piura,3045
7,16,Camila Neyra,Trujillo,2518
8,4,Jorge Salazar,Lima,2452


**Pregunta de cierre.** ¿Qué valor devuelve la subconsulta y cuántos clientes lo superan? ¿Por qué no debe compararse con el promedio por pedido del ejercicio 4?

**Respuesta:** La subconsulta devuelve el promedio del monto acumulado de los clientes que realizaron al menos un pedido. Según los resultados de la celda 12, son 9 los clientes que superan dicho promedio. No debe compararse con el promedio por pedido del ejercicio 4 porque son indicadores diferentes: el primero calcula el promedio de las compras acumuladas por cliente, mientras que el segundo calcula el importe promedio de cada pedido individual. Por lo tanto, ambos permiten analizar aspectos distintos del comportamiento de compra.


## Conclusiones

### Conclusiones técnicas

1. El uso de INNER JOIN, LEFT JOIN, RIGHT JOIN y FULL OUTER JOIN permitió relacionar las tablas de clientes, pedidos y productos mediante sus identificadores. Se comprendió que cada tipo de JOIN devuelve resultados diferentes según las coincidencias entre las tablas, facilitando la identificación de registros relacionados y sin correspondencia.

2. Las cláusulas WHERE, GROUP BY y HAVING permitieron filtrar, agrupar y analizar los registros de manera organizada. Además, las funciones COUNT, SUM y AVG facilitaron el cálculo de cantidades, ingresos y promedios, demostrando la importancia de utilizar correctamente las consultas SQL para obtener resultados precisos.

3. Las subconsultas y las vistas permitieron realizar análisis más completos y reutilizar consultas SQL sin repetir todo el código. Se comprobó que una vista almacena la definición de una consulta y facilita el acceso a información resumida, mejorando la organización y el mantenimiento de las consultas.

### Conclusiones de negocio

1. El análisis permitió identificar que, de los 24 clientes registrados, 18 realizaron compras y 6 no tienen pedidos. Arequipa y Piura concentran la mayor cantidad de clientes sin compras, con 2 cada una. Esta información puede ayudar al área comercial a desarrollar estrategias de seguimiento y promociones para incentivar futuras compras.

2. Se identificó que las categorías Celulares y Cómputo generaron ingresos de S/ 18 805 y S/ 17 662, respectivamente, superando el umbral de S/ 10 000. Estos resultados permiten reconocer las categorías con mayores ingresos y pueden servir como referencia para tomar decisiones relacionadas con inventarios, promociones y estrategias de venta.

3. El análisis de los clientes mostró que Patricia Zevallos registró el mayor monto acumulado con S/ 7123.00, mientras que Diego Alva obtuvo el mayor promedio por pedido con S/ 2939.50. Estos resultados permiten identificar clientes destacados y evaluar posibles beneficios comerciales. Sin embargo, antes de tomar decisiones, es importante considerar los márgenes de ganancia, los costos y la frecuencia de compra para asegurar que las estrategias sean rentables.
